In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset (adjust the filename if needed)
df = pd.read_csv('news.csv')

# Expecting columns 'text' and 'label' (change if your CSV uses different names)
# Convert boolean labels to integers if necessary
if df['label'].dtype == 'bool':
    df['label'] = df['label'].astype(int)

# Split the data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print("Data is ready for tokenization.")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

# Model checkpoint (TinyBERT) - change if you prefer a different checkpoint
model_checkpoint = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Dataset wrapper
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, padding=True, truncation=True, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
val_dataset = NewsDataset(val_texts, val_labels, tokenizer)

# Load model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Training arguments (tune these for your environment)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Trainer ready. Call trainer.train() in the next cell to start training.")


In [ ]:
# Start training
trainer.train()


In [ ]:
# Evaluate the model on the validation set
evaluation_results = trainer.evaluate()
print(evaluation_results)
